In [1]:
# ================================================================
#  LCCN-R4 - STRICTLY TRAINING-ONLY LEXICON EXPERIMENT  [CPU, ~2-3 h]
#  Rebuilds the noisy-to-clean map restricted to mappings whose noisy
#  form is attested in the TRAINING partition, freezes it, rebuilds
#  the lexicon-dependent arms (keel, lccn_base, lccn), and evaluates
#  them with the frozen classical anchors against the full-lexicon
#  arms from the certified NB1 splits. The emotion function F2E
#  derives from synset annotation, not from any evaluation comment,
#  and is unchanged; only the surface-form map is restricted.
#  Phonetic matchers do not consult the map and are unaffected.
#  Expected outcome per the companion bound: differences ~0.001.
#  Add Input: paper1-nb1-lccn-final (output: pipeline.py + splits),
#             paper1-nb2-lccn-ml (output: trainer.py),
#             final-emotion-dataset, gold-ds.
#  Accelerator None. Internet ON. Save & Run All (Commit).
# ================================================================
import subprocess, sys
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode
sh('pip install -q regex emoji wordfreq indic-nlp-library')
print("python", sys.version.split()[0])


+ pip install -q regex emoji wordfreq indic-nlp-library
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 84.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 5.5 MB/s eta 0:00:00
python 3.12.13


In [2]:
# ================================================================
#  CELL 1 - CERTIFIED PIPELINE (v5.3, selected explicitly)
# ================================================================
import shutil, importlib, sys, glob, re, os
for p in ("/kaggle/working/pipeline.py",):
    if os.path.exists(p): os.remove(p)
shutil.rmtree("/kaggle/working/__pycache__", ignore_errors=True)
sys.modules.pop("pipeline", None)
cand = sorted(glob.glob("/kaggle/input/**/pipeline.py", recursive=True))
chosen = None
for p in cand:
    m = re.search(r'PIPELINE_VERSION\s*=\s*"([^"]+)"',
                  open(p, encoding="utf-8", errors="replace").read())
    v = m.group(1) if m else "?"
    print(f"  v{v:<5} {p}")
    if v == "5.3" and chosen is None: chosen = p
assert chosen, "no v5.3 pipeline.py among the inputs"
shutil.copy(chosen, "/kaggle/working/pipeline.py")
if "/kaggle/working" not in sys.path: sys.path.insert(0, "/kaggle/working")
import pipeline
FP = pipeline.fingerprint()
assert pipeline.PIPELINE_VERSION == "5.3" and FP == "6a536dd437ba85b0"


  v5.3   /kaggle/input/notebooks/tejaswinina/paper1-nb1-lccn-final/pipeline.py
pipeline v5.3  sha256:6a536dd437ba85b0  SEGMENT=True


In [3]:
# ================================================================
#  CELL 2 - SHARED SETUP + TRAIN-ATTESTED RESTRICTION OF THE MAP
# ================================================================
import importlib
importlib.reload(pipeline)
from pipeline import *
import pandas as pd
from collections import Counter

df, ID, TXT, LAB, RATERS = load_corpus()
norm_map, F2E, LEX = build_keel()
try:
    from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    _n = IndicNormalizerFactory().get_normalizer("kn")
    _n.normalize("\u0c95"); pipeline._indic_norm = _n; print("IndicNLP: FULL")
except Exception:
    import unicodedata
    class _F:
        def normalize(self, s): return unicodedata.normalize("NFC", s)
    pipeline._indic_norm = _F(); print("IndicNLP: FALLBACK(NFC)")

df["text_raw"] = df[TXT].map(lambda t: " ".join(process_comment(t)[0]))
SPLIT, TEST_HASH = make_split(df, ID, LAB)
assert TEST_HASH == "e2d6e1f7d564cace", f"split drift: {TEST_HASH}"
train = df[df[ID].isin(SPLIT["train"])]
evidence = LabelEvidence(train, TXT, LAB)
TRAIN_FREQ = Counter()
for t in train.text_raw: TRAIN_FREQ.update(w.lower() for w in t.split())
pipeline.TRAIN_FREQ = TRAIN_FREQ

attested = set(TRAIN_FREQ)
nm_train = {k: v for k, v in norm_map.items() if k in attested}
print(f"noisy-to-clean map: full {len(norm_map):,} -> "
      f"train-attested {len(nm_train):,} "
      f"({100*len(nm_train)/len(norm_map):.2f}% retained)")
print("F2E (synset-derived, no corpus labels) unchanged:",
      f"{len(F2E):,} entries")


corpus: 40,000 -> 40,000 after audit (dedup + conflict removal)
majority
disgust    10000
happy      10000
sad        10000
anger      10000
  KEEL cascade | tier1 126,567 | tier2 0 | tier3 603 | emotions 174,425 (3,658 ambiguous)
    multi-word entries skipped (word-level lookup): 274,777
    negated phrases kept for the context-aware veto: 706 | va_id misses 0
  KEEL entries: 126,567 | seg vocab: 12,326 | form->emotion: 174,425
IndicNLP: FULL
train  28,000
val     6,000
test    6,000
leakage check passed | test-set id hash: e2d6e1f7d564cace
noisy-to-clean map: full 126,567 -> train-attested 2,529 (2.00% retained)
F2E (synset-derived, no corpus labels) unchanged: 174,425 entries


In [4]:
# ================================================================
#  CELL 3 - REBUILD THE LEXICON-DEPENDENT ARMS UNDER THE
#           TRAIN-ONLY MAP
# ================================================================
from tqdm.auto import tqdm; tqdm.pandas()
veto = make_veto(F2E, LEX)

ARMS_T = {}
ARMS_T["keel_trainonly"] = lambda t, m=nm_train: " ".join(
    m.get(w.lower(), w) for w in process_comment(t)[0])
o_base = LCCN(nm_train, evidence, veto, translit=None, lex=LEX,
              use_gtest=False, use_cluster=False, use_ortho=False)
o_full = LCCN(nm_train, evidence, veto, translit=None, lex=LEX)
ARMS_T["lccn_base_trainonly"] = o_base.normalize
ARMS_T["lccn_trainonly"]      = o_full.normalize

for name, fn in ARMS_T.items():
    df[f"text_{name}"] = df[TXT].progress_apply(fn); print("built", name)
o_full.report("lccn_trainonly")

for name in ARMS_T:
    ch = tot = 0
    for a, b in zip(df["text_raw"], df[f"text_{name}"]):
        ra, rb = a.lower().split(), b.lower().split()
        if len(ra) != len(rb): continue
        tot += len(ra); ch += sum(1 for u, v in zip(ra, rb) if u != v)
    print(f"{name:<22} tokens changed {100*ch/tot:5.2f}%")


  0%|          | 0/40000 [00:00<?, ?it/s]

built keel_trainonly


  0%|          | 0/40000 [00:00<?, ?it/s]

built lccn_base_trainonly


  0%|          | 0/40000 [00:00<?, ?it/s]

built lccn_trainonly
  lccn_trainonly: merged 5,976 of 54,694 distinct forms
    blocked_cluster                   35
    blocked_gtest                    259
    blocked_orthographic           3,710
    blocked_veto_emotion             111
    blocked_veto_length                3
    blocked_veto_negation             42
keel_trainonly         tokens changed 16.76%
lccn_base_trainonly    tokens changed 19.12%
lccn_trainonly         tokens changed 14.17%


In [5]:
# ================================================================
#  CELL 4 - CLASSICAL ANCHORS: TRAIN-ONLY vs FULL-LEXICON ARMS
#  trainer v1.4 byte-identical from the NB2 output; the full-lexicon
#  columns (keel, lccn_base, lccn) come from the certified NB1
#  splits, so the comparison is within one frozen split.
# ================================================================
import shutil, glob, os, sys, importlib
tc = sorted(glob.glob("/kaggle/input/**/trainer.py", recursive=True))
assert tc, "attach the paper1-nb2-lccn-ml output"
shutil.copy(tc[0], "/kaggle/working/trainer.py")
sys.modules.pop("trainer", None)
import trainer
importlib.reload(trainer)
from trainer import *
assert trainer.TRAINER_VERSION == "1.4"
trainer.fingerprint()

sd = [os.path.dirname(p) for p in
      glob.glob("/kaggle/input/**/splits/train.csv", recursive=True)]
assert sd, "attach the paper1-nb1-lccn-final output"
tr, va, te, HAVE = load_splits(sd[0])

lut = {c: dict(zip(df[ID].astype(str), df[f"text_{c}"])) for c in ARMS_T}
for f in (tr, va, te):
    ids = f["id"].astype(str)
    for c in ARMS_T:
        f[f"text_{c.lower()}"] = ids.map(lut[c])
        assert f[f"text_{c.lower()}"].notna().all(), f"id join failed: {c}"

ARMS = ["keel", "keel_trainonly", "lccn_base", "lccn_base_trainonly",
        "lccn", "lccn_trainonly"]
missing = [a for a in ARMS if f"text_{a}" not in tr.columns]
assert not missing, f"missing arms: {missing}"
res = run_grid(tr, va, te, ARMS, ["Logistic Regression", "SVM"],
               tag="lccn_r4_trainonly", resume=False)
r_lr, r_svm = res.iloc[0], res.iloc[1]
print(f"\n{'arm':<24}{'LR':>8}{'SVM':>8}{'mean':>8}")
for a in ARMS:
    m = (float(r_lr[a]) + float(r_svm[a])) / 2
    print(f"{a:<24}{float(r_lr[a]):>8.4f}{float(r_svm[a]):>8.4f}{m:>8.4f}")
print()
for full, only in (("keel", "keel_trainonly"),
                   ("lccn_base", "lccn_base_trainonly"),
                   ("lccn", "lccn_trainonly")):
    d = ((float(r_lr[only]) + float(r_svm[only]))
         - (float(r_lr[full]) + float(r_svm[full]))) / 2
    print(f"train-only minus full  {full:<10} {d:+.4f}")
print("\nDONE - transcribe only from the committed log.")


trainer v1.4  sha256:5b607248090bd26e
  BUDGET {'epochs': 10, 'lr': 2e-05, 'batch_size': 32, 'max_len': 96, 'warmup_ratio': 0.1, 'weight_decay': 0.01, 'patience': 3}
  DEEP   {'epochs': 8, 'lr': 0.001, 'batch_size': 64, 'max_len': 64, 'vocab': 40000, 'emb': 200, 'hid': 128, 'dropout': 0.3, 'patience': 2}
test-set id hash: e2d6e1f7d564cace
train  28,000  {'disgust': np.int64(7000), 'happy': np.int64(7000), 'sad': np.int64(7000), 'anger': np.int64(7000)}
val     6,000  {'anger': np.int64(1500), 'happy': np.int64(1500), 'sad': np.int64(1500), 'disgust': np.int64(1500)}
test    6,000  {'disgust': np.int64(1500), 'anger': np.int64(1500), 'sad': np.int64(1500), 'happy': np.int64(1500)}
arms found: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']

=== Logistic Regression / keel ===
  -> macro-F1 0.8019

=== Logistic Regression / keel_trainonly ===
  -> macro-F1 0.8022

=== Logi